# GPT-OSS Reference Profiling Notebook

Profile GPT-OSS with dummy BF16 weights and the forward path in `model.py`.

## Workflow

1. Edit the next cell only.
2. Run the preflight cell.
3. Run the single-workload cells.
4. Enable the sweep only after a safe single run.

## Decode Semantics

Strict-reference mode uses one prompt forward pass for prefill. If `generated_tokens > 0`, the first new token is sampled from the prefill logits. `decode_total` measures only the follow-on full-sequence passes for the remaining generated tokens.

## Note On Memory

This reference MoE path materializes large temporary expert tensors. On unified-memory systems such as DGX Spark, oversized runs can make the machine unresponsive instead of raising a clean CUDA OOM. Start small and use the preflight output as a guard.


In [ ]:
from dataclasses import asdict
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

CWD = Path.cwd().resolve()
REPO_ROOT = next(
    (path for path in (CWD, *CWD.parents) if (path / "pyproject.toml").exists()),
    CWD,
)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from profiling import (
    DEFAULT_GENERATED_SWEEP,
    DEFAULT_PREFILL_SWEEP,
    TimingConfig,
    WorkloadConfig,
    format_bytes,
    plot_level0_breakdown,
    plot_level1_breakdown,
    plot_level2_heatmap,
    plot_workload_grid,
    preflight_report,
    run_workload,
    run_workload_sweep,
    summarize_results,
)

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)


In [ ]:
# EDIT THIS CELL ONLY

# MODEL SETTINGS
# Values below are the actual numbers that will be used.
# These start from the gpt-oss-20b defaults.
# For gpt-oss-120b, the reference preset differs at least in:
#   num_hidden_layers = 36
#   num_experts = 128

PRESET_NAME = "gpt-oss-20b"  # Label used in result tables

ARCHITECTURE = {
    # Core model
    "num_hidden_layers": 24,
    "vocab_size": 201088,
    "hidden_size": 2880,
    "intermediate_size": 2880,

    # Mixture of experts
    "num_experts": 32,
    "experts_per_token": 4,
    "swiglu_limit": 7.0,

    # Attention
    "head_dim": 64,
    "num_attention_heads": 64,
    "num_key_value_heads": 8,
    "sliding_window": 128,

    # Context and RoPE
    "initial_context_length": 4096,
    "rope_theta": 150000.0,
    "rope_scaling_factor": 32.0,
    "rope_ntk_alpha": 1.0,
    "rope_ntk_beta": 32.0,
}

# WORKLOAD SETTINGS
DEVICE = "cuda"
SEED = 0

SINGLE_WORKLOAD = {
    "prefill_tokens": 1,
    "generated_tokens": 1,
}

# TIMING SETTINGS
TIMING = TimingConfig(
    warmup_iters=0,
    measure_iters=1,
    seed=SEED,
    enable_torch_profiler=False,
    trace_output_dir="traces",
)

# SWEEP SETTINGS
RUN_SWEEP = False
SWEEP_PREFILL = list(DEFAULT_PREFILL_SWEEP)
SWEEP_GENERATED = list(DEFAULT_GENERATED_SWEEP)

# OPTIONAL FULL LEVEL-2 OUTPUT
SHOW_FULL_LEVEL2 = False
EXPORT_FULL_LEVEL2_CSV = False
FULL_LEVEL2_CSV_PATH = "outputs/level2_full.csv"


In [ ]:
from model import ModelConfig

arch_cfg = ModelConfig(**ARCHITECTURE)
single_workload = WorkloadConfig(**SINGLE_WORKLOAD)
report = preflight_report(arch_cfg, single_workload, device=DEVICE)

arch_table = pd.DataFrame(
    [{"field": key, "value": value} for key, value in asdict(arch_cfg).items()]
)
preflight_table = pd.DataFrame(
    [{"metric": key, "value": value} for key, value in report.items()]
)
preflight_table["pretty_value"] = preflight_table.apply(
    lambda row: format_bytes(row["value"])
    if isinstance(row["value"], int) and ("bytes" in row["metric"])
    else row["value"],
    axis=1,
)

important_metrics = [
    "device_total_bytes",
    "device_free_bytes",
    "dense_weight_bytes",
    "peak_runtime_component",
    "peak_runtime_bytes",
    "moe_mlp1_temp_bytes",
    "moe_mlp2_temp_bytes",
    "logits_bytes",
    "estimated_peak_bytes",
    "fits_free_memory",
]

display(arch_table)
display(
    preflight_table[preflight_table["metric"].isin(important_metrics)][
        ["metric", "pretty_value"]
    ]
)

if report.get("fits_free_memory") is False:
    print("Preflight says this run is unsafe. Reduce model size or token counts before running the model.")
else:
    print("Preflight passed for the current workload.")


In [ ]:
if DEVICE == "cuda" and report.get("fits_free_memory") is False:
    raise RuntimeError(
        "Preflight failed. Reduce model size or token counts before running the model."
    )

single_rows = run_workload(
    arch_cfg,
    single_workload,
    TIMING,
    device=DEVICE,
    preset_name=PRESET_NAME,
)
single_summary = summarize_results(single_rows)

trace_path = single_rows.attrs.get("trace_path")
if trace_path:
    print(f"Chrome trace written to: {trace_path}")


In [ ]:
level1_table = single_summary["level1"][[
    "phase",
    "scope_name",
    "mean_duration_ms",
]].sort_values(["phase", "scope_name"]).reset_index(drop=True)

level2_table = single_summary["level2"][[
    "phase",
    "component",
    "scope_name",
    "mean_duration_ms",
]].sort_values(["phase", "component", "scope_name"]).reset_index(drop=True)

display(single_summary["level0_metrics"])
display(level1_table)
display(level2_table)


In [ ]:
if SHOW_FULL_LEVEL2:
    full_level2 = single_summary["level2_full"].sort_values(
        ["phase", "component", "layer_idx", "scope_name"]
    ).reset_index(drop=True)
    display(full_level2)

    if EXPORT_FULL_LEVEL2_CSV:
        output_path = REPO_ROOT / FULL_LEVEL2_CSV_PATH
        output_path.parent.mkdir(parents=True, exist_ok=True)
        full_level2.to_csv(output_path, index=False)
        print(f"Wrote {output_path}")
else:
    print("Set SHOW_FULL_LEVEL2 = True in the config cell to see the per-layer level-2 breakdown.")


In [ ]:
plot_level0_breakdown(single_summary["level0_metrics"])
plot_level1_breakdown(single_summary["level1"], phase="combined")
plot_level2_heatmap(
    single_summary["level2_full"],
    phase="prefill",
    component="attention",
    prefill_tokens=SINGLE_WORKLOAD["prefill_tokens"],
    generated_tokens=SINGLE_WORKLOAD["generated_tokens"],
)
plot_level2_heatmap(
    single_summary["level2_full"],
    phase="prefill",
    component="mlp",
    prefill_tokens=SINGLE_WORKLOAD["prefill_tokens"],
    generated_tokens=SINGLE_WORKLOAD["generated_tokens"],
)


In [ ]:
if not RUN_SWEEP:
    print("Set RUN_SWEEP = True in the config cell after a safe single run.")
else:
    sweep_rows = run_workload_sweep(
        arch_cfg,
        prefill_tokens=SWEEP_PREFILL,
        generated_tokens=SWEEP_GENERATED,
        timing_cfg=TIMING,
        device=DEVICE,
        preset_name=PRESET_NAME,
    )
    sweep_summary = summarize_results(sweep_rows)

    level1_sweep_table = sweep_summary["level1"][[
        "prefill_tokens",
        "generated_tokens",
        "phase",
        "scope_name",
        "mean_duration_ms",
    ]].sort_values(["prefill_tokens", "generated_tokens", "phase", "scope_name"]).reset_index(drop=True)

    display(
        sweep_summary["level0_metrics"]
        .sort_values(["prefill_tokens", "generated_tokens"])
        .reset_index(drop=True)
    )
    display(level1_sweep_table)

    plot_workload_grid(sweep_summary["level0_metrics"], value="workload_total")
    plot_workload_grid(sweep_summary["level0_metrics"], value="avg_ms_per_new_token")
    plot_level1_breakdown(sweep_summary["level1"], phase="combined")
